# Summit-Sim: Complete E2E Workflow Demo

Demonstrates the full Summit-Sim workflow:
1. **Scenario Generation** - AI generates wilderness rescue scenario
2. **Simulation** - Student navigates scenario with human-in-the-loop
3. **Debrief** - Performance analysis and learning report

All phases are traced to MLflow for visibility.

## 1. Setup & Imports

In [1]:
import warnings

# Suppress autologging warnings
warnings.filterwarnings("ignore", message=".*autologging.*")
warnings.filterwarnings("ignore", message=".*LangChain.*")

from summit_sim.agents.generator import generate_scenario
from summit_sim.graphs.simulation import create_simulation_graph
from summit_sim.schemas import HostConfig, generate_scenario_id
from summit_sim.settings import settings
from summit_sim.tracing import enable_tracing, simulation_session

# Initialize MLflow tracing
enable_tracing()

print(f"✓ MLflow: {settings.mlflow_tracking_uri}")
print(f"✓ Experiment: {settings.mlflow_experiment_name}")
print(f"✓ Model: {settings.default_model}")
print(f"✓ API Key: {bool(settings.openrouter_api_key)}")

✓ MLflow: https://mlflow.bhamm-lab.com
✓ Experiment: summit-sim
✓ Model: google/gemini-3.1-flash-lite-preview
✓ API Key: True


## 2. Phase 1: Scenario Generation

Generate a complete wilderness rescue scenario from minimal host configuration.

In [2]:
# Configuration
config = HostConfig(
    num_participants=3,
    activity_type="hiking",
    difficulty="med",
    class_id="demo-class-2024",
)

print(
    f"Generating: {config.activity_type} scenario for {config.num_participants} participants"
)
print("-" * 60)

# Generate scenario
scenario = await generate_scenario(config)
scenario_id = generate_scenario_id()

print(f"✓ Title: {scenario.title}")
print(f"✓ Setting: {scenario.setting}")
print(f"✓ Patient: {scenario.patient_summary}")
print(f"✓ Turns: {len(scenario.turns)}")
print(f"✓ Scenario ID: {scenario_id}")
print("\nLearning Objectives:")
for obj in scenario.learning_objectives:
    print(f"  • {obj}")

Generating: hiking scenario for 3 participants
------------------------------------------------------------
✓ Title: The Alpine Slip
✓ Setting: Rugged alpine hiking trail at 8,000 feet elevation. Mid-afternoon, temperature is 50°F (10°C) and dropping with light rain starting.
✓ Patient: 28-year-old male hiker, fell approximately 10 feet onto a rocky slope. Visible deformity of the right ankle, pale complexion, anxious demeanor, shivering.
✓ Turns: 4
✓ Scenario ID: scn-38d090ba

Learning Objectives:
  • Perform a systematic patient assessment (ABCDE) despite distracting injuries.
  • Prioritize environmental management (hypothermia prevention) in a wilderness setting.
  • Apply appropriate wilderness splinting techniques (splint as it lies).


Trace(trace_id=tr-43eb8e3244ea2f3764f57a1953c07f1f)

### Display Scenario Structure

In [3]:
for turn in scenario.turns:
    marker = "(START)" if turn.is_starting_turn else ""
    print(f"\n{'=' * 60}")
    print(f"Turn {turn.turn_id} {marker}")
    print(f"{'=' * 60}")
    print(
        turn.narrative_text[:200] + "..."
        if len(turn.narrative_text) > 200
        else turn.narrative_text
    )
    print("\nChoices:")
    for choice in turn.choices:
        mark = "✓" if choice.is_correct else " "
        next_turn = (
            "END" if choice.next_turn_id is None else f"Turn {choice.next_turn_id}"
        )
        print(f"  [{mark}] {choice.description[:60]}... → {next_turn}")


Turn 0 (START)
You are hiking with two friends when Alex, 28, falls while traversing a steep, rocky section. He is now on the ground, holding his right ankle, loudly expressing intense pain. The ankle is visibly swo...

Choices:
  [ ] Rush to splint the deformed ankle immediately to stop the pa... → Turn 1
  [✓] Perform a quick Primary Survey (ABCDE), check vital signs, a... → Turn 1

Turn 1 
Your survey reveals Alex is shivering, his skin is pale and clammy to the touch, and his heart rate is elevated (110 bpm). He is awake but anxious. The rain is intensifying. What is your next move?

Choices:
  [ ] Ask your group to help carry Alex down the trail to the trai... → Turn 2
  [✓] Insulate him from the ground using packs, get him under a ra... → Turn 2

Turn 2 
Alex is now covered and lying on top of your packs to insulate him from the cold ground. His condition is stable, but the ankle deformity is severe. How do you handle the injury?

Choices:
  [ ] Attempt to reduce (realign) the a

## 3. Phase 2: Simulation

Run the interactive simulation with human-in-the-loop.
The graph will pause at each turn for you to make a choice.

In [4]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# Create graph with memory
memory = InMemorySaver()
graph = create_simulation_graph(checkpointer=memory)

# Initialize state
initial_state = {
    "scenario_draft": scenario,
    "current_turn_id": scenario.starting_turn_id,
    "transcript": [],
    "is_complete": False,
    "key_learning_moments": [],
    "last_selected_choice": None,
    "simulation_result": None,
    "scenario_id": scenario_id,
    "class_id": config.class_id,
    "debrief_report": None,
}

print("✓ Graph initialized")
print(f"✓ Starting at turn {initial_state['current_turn_id']}")

✓ Graph initialized
✓ Starting at turn 0


In [5]:
# Run simulation with MLflow session tracking
from summit_sim.tracing import log_simulation_results

with simulation_session(config, scenario_id=scenario_id) as (session_id, graph_config):
    print(f"\n📋 Session ID: {session_id}")
    print("🎮 Starting simulation...\n")
    print("=" * 60)
    print("INTERACTIVE SIMULATION")
    print("=" * 60 + "\n")

    current_state = initial_state
    turn_count = 0
    simulation_complete = False
    first_turn = True

    while not simulation_complete:
        turn_count += 1

        # Get the current turn from the graph state
        current_turn_id = current_state["current_turn_id"]
        current_turn = scenario.get_turn(current_turn_id)

        if current_turn is None:
            print(f"Error: Turn {current_turn_id} not found")
            break

        # Display turn
        print(f"\n--- TURN {turn_count} ---\n")
        print(current_turn.narrative_text)
        print("\nChoices:")
        for i, choice in enumerate(current_turn.choices, 1):
            print(f"  {i}. {choice.description}")

        # Get user input
        while True:
            try:
                sel = int(input("\nSelect choice (number): ")) - 1
                if 0 <= sel < len(current_turn.choices):
                    break
                print("Invalid selection")
            except ValueError:
                print("Please enter a number")

        selected_choice = current_turn.choices[sel]

        # Process the turn using ainvoke to maintain tracing context
        if first_turn:
            # First turn: pass full state to initialize the graph
            current_state = await graph.ainvoke(
                initial_state,
                graph_config,
            )
            first_turn = False
        else:
            # Subsequent turns: use Command to resume
            current_state = await graph.ainvoke(
                Command(resume={"choice_id": selected_choice.choice_id}),
                graph_config,
            )

        # Check if complete
        if current_state.get("is_complete"):
            simulation_complete = True

        # Safety limit
        if turn_count > 10:
            print("\nSafety limit reached")
            break

    print("\n" + "=" * 60)
    print("SIMULATION COMPLETE")
    print("=" * 60)

    # Log results
    log_simulation_results(
        transcript=current_state["transcript"],
        is_complete=current_state["is_complete"],
        key_learning_moments=current_state["key_learning_moments"],
    )
    print("\n✓ Results logged to MLflow")
    print(f"✓ Total turns: {len(current_state['transcript'])}")
    print(f"✓ Learning moments: {len(current_state['key_learning_moments'])}")


📋 Session ID: d13ba648-3679-41fb-9ba4-a4cd5a1d0911
🎮 Starting simulation...

INTERACTIVE SIMULATION


--- TURN 1 ---

You are hiking with two friends when Alex, 28, falls while traversing a steep, rocky section. He is now on the ground, holding his right ankle, loudly expressing intense pain. The ankle is visibly swollen and deformed. It is starting to rain, and the temperature is noticeably dropping. What is your first priority?

Choices:
  1. Rush to splint the deformed ankle immediately to stop the pain.
  2. Perform a quick Primary Survey (ABCDE), check vital signs, and initiate environmental protection measures before focusing on the ankle.


2026/03/23 22:13:20 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f4c3c917510> at 0x7f4beff182c0> was created in a different Context
2026/03/23 22:13:20 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f4c3c917510> at 0x7f4beff83600> was created in a different Context
2026/03/23 22:13:20 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f4c3c917510> at 0x7f4beff11400> was created in a different Context



--- TURN 2 ---

You are hiking with two friends when Alex, 28, falls while traversing a steep, rocky section. He is now on the ground, holding his right ankle, loudly expressing intense pain. The ankle is visibly swollen and deformed. It is starting to rain, and the temperature is noticeably dropping. What is your first priority?

Choices:
  1. Rush to splint the deformed ankle immediately to stop the pain.
  2. Perform a quick Primary Survey (ABCDE), check vital signs, and initiate environmental protection measures before focusing on the ankle.


2026/03/23 22:13:21 WARNING mlflow.tracing.fluent: Failed to start span LangGraph: 'NoneType' object has no attribute 'set_span_type'. For full traceback, set logging level to debug.
Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
2026/03/23 22:13:21 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f4c3c917510> at 0x7f4beffd8b40> was created in a different Context
2026/03/23 22:13:37 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f4c3c917510> at 0x7f4beff76240> was created in a different Context
2026/03/23 22:13:37 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVa


--- TURN 3 ---

Your survey reveals Alex is shivering, his skin is pale and clammy to the touch, and his heart rate is elevated (110 bpm). He is awake but anxious. The rain is intensifying. What is your next move?

Choices:
  1. Ask your group to help carry Alex down the trail to the trailhead, 3 miles away.
  2. Insulate him from the ground using packs, get him under a rain tarp, and check his distal sensation/motor/pulse (CMS).


...

2026/03/23 22:14:01 WARNING mlflow.tracing.fluent: Failed to start span LangGraph: 'NoneType' object has no attribute 'set_span_type'. For full traceback, set logging level to debug.
Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
Deserializing unregistered type summit_sim.schemas.ChoiceOption from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ChoiceOption')]
Deserializing unregistered type summit_sim.schemas.SimulationResult from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'SimulationResult')]
2026/03/23 22:14:01 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f4c3c917510> a


--- TURN 5 ---

The ankle is effectively splinted, and Alex seems slightly warmer. You have signaled for help using your Personal Locator Beacon (PLB). What is your final strategy?

Choices:
  1. Start walking slowly down the trail, supporting Alex, while calling for help on a radio.
  2. Set up a permanent shelter, maintain patient warmth, monitor vitals, and wait for the SAR team you have notified via PLB.


Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
Deserializing unregistered type summit_sim.schemas.ChoiceOption from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ChoiceOption')]
Deserializing unregistered type summit_sim.schemas.SimulationResult from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'SimulationResult')]
2026/03/23 22:14:21 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f4c3c917510> at 0x7f4befe01280> was created in a different Context
2026/03/23 22:14:26 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVa


SIMULATION COMPLETE

✓ Results logged to MLflow
✓ Total turns: 4
✓ Learning moments: 8
🏃 View run sim-demo-class-2024-hiking-3p-med at: https://mlflow.bhamm-lab.com/#/experiments/7/runs/cfbf30c7bc144b5d99bcd0fdd82acea2
🧪 View experiment at: https://mlflow.bhamm-lab.com/#/experiments/7


[Trace(trace_id=tr-3991e9b1cde2366ed17e1c83f671db7d), Trace(trace_id=tr-276b392437edd62c71082c34a1ec3d07), Trace(trace_id=tr-6696c4c4db808d24e3525746f96192b6), Trace(trace_id=tr-98ba61e7ddc55c3a3dc20b9615f02c34), Trace(trace_id=tr-d8875b06b1cb035cc9d3e1a522a0f266)]

## 4. Phase 3: Debrief

View the comprehensive performance report generated automatically by the simulation graph.

In [6]:
debrief_report = current_state["debrief_report"]

print("=" * 60)
print("DEBRIEF REPORT")
print("=" * 60)

print(f"\n📊 Final Score: {debrief_report.final_score:.1f}%")
status_emoji = "✅" if debrief_report.completion_status == "pass" else "❌"
print(f"{status_emoji} Status: {debrief_report.completion_status.upper()}")

print("\n📝 Summary:")
print(debrief_report.summary)

if debrief_report.key_mistakes:
    print("\n⚠️ Key Mistakes:")
    for mistake in debrief_report.key_mistakes:
        print(f"  • {mistake}")

if debrief_report.strong_actions:
    print("\n💪 Strong Actions:")
    for action in debrief_report.strong_actions:
        print(f"  • {action}")

if debrief_report.teaching_points:
    print("\n📚 Teaching Points:")
    for point in debrief_report.teaching_points:
        print(f"  • {point}")

if debrief_report.best_next_actions:
    print("\n🎯 Recommendations:")
    for rec in debrief_report.best_next_actions:
        print(f"  • {rec}")

DEBRIEF REPORT

📊 Final Score: 100.0%
✅ Status: PASS

📝 Summary:
You demonstrated outstanding clinical decision-making during 'The Alpine Slip' simulation. You showed a disciplined approach by adhering to the ABCDE patient assessment protocol, which ensured that vital systemic stability and thermoregulation were prioritized over the visibly painful, yet secondary, ankle injury. Your actions effectively managed the patient's physiological and psychological needs throughout the scenario. Exceptional work.

⚠️ Key Mistakes:
  • None identified in this simulation.

💪 Strong Actions:
  • Consistently prioritized ABCDE assessment before attending to extremity injuries.
  • Demonstrated expert adherence to the 'splint as it lies' principle to prevent further neurovascular injury.
  • Correctly prioritized environmental management, demonstrating an understanding of the relationship between trauma, shock, and hypothermia even in mild temperatures.
  • Made the safest decision for the patient by

## 5. Results Summary

Complete transcript and learning moments from the simulation.

In [7]:
print("\n" + "=" * 60)
print("COMPLETE TRANSCRIPT")
print("=" * 60 + "\n")

for i, entry in enumerate(current_state["transcript"], 1):
    status = "✓" if entry["was_correct"] else "✗"
    print(f"Turn {i} {status}")
    print(f"  Situation: {entry['turn_narrative'][:80]}...")
    print(f"  Choice: {entry['choice_description']}")
    print(f"  Feedback: {entry['feedback'][:100]}...")
    print()


COMPLETE TRANSCRIPT

Turn 1 ✓
  Situation: You are hiking with two friends when Alex, 28, falls while traversing a steep, r...
  Choice: Perform a quick Primary Survey (ABCDE), check vital signs, and initiate environmental protection measures before focusing on the ankle.
  Feedback: Excellent judgment. By prioritizing a full ABCDE assessment and environmental protection, you have e...

Turn 2 ✓
  Situation: Your survey reveals Alex is shivering, his skin is pale and clammy to the touch,...
  Choice: Insulate him from the ground using packs, get him under a rain tarp, and check his distal sensation/motor/pulse (CMS).
  Feedback: Great call. Prioritizing thermoregulation and assessing distal CMS before considering any movement i...

Turn 3 ✓
  Situation: Alex is now covered and lying on top of your packs to insulate him from the cold...
  Choice: Splint the ankle 'as it lies' using your sleeping pad foam and extra clothing for padding, and re-check CMS.
  Feedback: Excellent judgment. 

In [8]:
print("\n" + "=" * 60)
print("KEY LEARNING MOMENTS")
print("=" * 60 + "\n")

for i, moment in enumerate(current_state["key_learning_moments"], 1):
    print(f"{i}. {moment}")


KEY LEARNING MOMENTS

1. Prioritizing the primary survey (ABCDE) is essential to identify life-threatening conditions (like shock) that may be masked by the emotional distress of a dramatic extremity injury.
2. In wilderness environments, hypothermia prevention is not just a comfort measure; it is a critical clinical intervention that must be initiated early to preserve the patient’s metabolic function.
3. Hypothermia is a significant risk in wilderness settings; always insulate the patient from the cold ground early to maintain core temperature.
4. When managing musculoskeletal injuries, always document distal Circulation, Motor, and Sensory (CMS) function before and after any splinting or attempt to move the patient.
5. Standard wilderness protocol dictates that you should splint fractures in the position found unless there is an absolute loss of distal circulation that cannot be corrected.
6. Always perform a neurovascular assessment (CMS: Circulation, Motor, Sensory) both before a

---
## ✅ Complete Workflow Demo

All phases executed:
- ✓ Scenario generated with AI
- ✓ Simulation completed with human-in-the-loop
- ✓ Debrief report generated
- ✓ All traces logged to MLflow

View traces in MLflow UI at your configured tracking URI.